# M1 — Cleaning and feature engineering

Repository-upload draft, updated 19 October 2026. Owner: Jin Zihao (M1).

This notebook embeds its M1 processing functions; it does not import or change `src/prep.py` or depend on unpublished M1 scripts. It requires local source data and Python dependencies: pandas, numpy, scipy, scikit-learn and lxml. Original tested environment: Python 3.12; use the group-approved environment.

The accompanying dictionary documents the **48 M1 clean-data fields**, not the complete SQL/complaint modelling extract. Recorded missing counts refer to 15,155 retained listings before imputation.

Eight historically flagged rows are retained by team decision; review closed. Condition features were tested by M5 and omitted from final XGBoost. Final preprocessing remains owned by M5. The remote main branch inspected on 19 October contained prep v4; do not substitute this notebook for the privately shared v5.

Keep source data locally under `data/raw/`; use the M1 handover or full cars.csv/cars.xlsx. Do not upload source data merely to make this notebook run. Default outputs go to `data/clean` and `data/diagnostics`; execution regenerates the original M1 split artifacts, so run in a local working copy and preserve M5 final split/fold files.


In [1]:
from pathlib import Path
import json, pandas as pd
# Optional: point PROJECT_ROOT at a local checkout containing data/raw/.
PROJECT_ROOT = None
if PROJECT_ROOT is None:
    candidates = [Path.cwd(), *Path.cwd().parents]
    ROOT = next((p for p in candidates if (p / 'data/raw').is_dir()), None)
    if ROOT is None:
        ROOT = next((p for p in candidates if (p / '.git').exists()), Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd())
else:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()
rawdir = ROOT / 'data/raw'
full_source = (rawdir / 'cars.csv').exists() or (rawdir / 'cars.xlsx').exists()
snapshot = (rawdir / 'electric_source_snapshot.csv').exists() and (rawdir / 'source_profile.json').exists()
if not (full_source or snapshot):
    raise FileNotFoundError('Place cars.csv/cars.xlsx in local data/raw/, or copy electric_source_snapshot.csv AND source_profile.json from the M1 handover. Data are not bundled in this notebook.')
print('Local source files found. Cleaning functions are defined below.')


Local source files found. Cleaning functions are defined below.


In [2]:
# Embedded M1 implementation: condition_features.py
"""Fixed, transparent M1 condition rules. No price or population statistics used.
All thresholds are project assumptions, not validated mechanical standards.
"""
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

CONDITION_BINARY = ['low_mileage_for_age_flag', 'near_new_flag', 'clean_history_flag']
CONDITION_NUMERIC = ['condition_score', 'battery_wear_proxy']
CONDITION_CATEGORY = ['age_band', 'mileage_band']
CONDITION_FEATURES = CONDITION_BINARY + CONDITION_NUMERIC + CONDITION_CATEGORY

def add_condition_features(frame):
    out = frame.copy()
    age = pd.to_numeric(out['age'], errors='coerce')
    miles = pd.to_numeric(out['mileage'], errors='coerce')
    accident = pd.to_numeric(out['accidents_or_damage'], errors='coerce')
    valid = age.ge(0) & miles.ge(0) & age.notna() & miles.notna()
    annual = miles / age.clip(lower=0.5)
    out['low_mileage_for_age_flag'] = np.where(valid, annual.lt(7500).astype(float), np.nan)
    # Three-valued logic: known failure=0; known success=1; otherwise unknown.
    near = pd.Series(np.nan, index=out.index)
    near.loc[age.gt(1) | miles.ge(5000)] = 0.0
    near.loc[valid & age.le(1) & miles.lt(5000)] = 1.0
    out['near_new_flag'] = near
    out['clean_history_flag'] = accident.map({0.0: 1.0, 1.0: 0.0})
    # Do not reward missing history or incomplete condition information.
    out['condition_score'] = out[CONDITION_BINARY].sum(axis=1, min_count=3)
    out['age_band'] = pd.cut(age.where(age.ge(0)), [-np.inf,1,3,5,np.inf], labels=['0-1','2-3','4-5','6+']).astype(object)
    out['mileage_band'] = pd.cut(miles.where(miles.ge(0)), [0,10000,30000,60000,np.inf], right=False, labels=['0-<10k','10k-<30k','30k-<60k','60k+']).astype(object)
    # Cumulative use exposure in 10,000-mile units, NOT battery SOH or degradation %.
    out['battery_wear_proxy'] = miles.where(miles.ge(0)) / 10000.0
    return out

class ConditionFeatures(BaseEstimator, TransformerMixin):
    """Place before M5 preprocessing in an sklearn Pipeline; no fitted thresholds."""
    def fit(self, X, y=None):
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def transform(self, X):
        return add_condition_features(X)


# Embedded M1 implementation: source_loader.py
"""Scan the full supplied source when available; otherwise use the audited snapshot."""
from pathlib import Path
from zipfile import ZipFile
from lxml import etree
import pandas as pd
import json,hashlib,collections

def load_source(root):
    root=Path(root); rawdir=root/'data/raw'
    xlsx=rawdir/'cars.xlsx'; csv=rawdir/'cars.csv'
    if xlsx.exists():
        ns='{http://schemas.openxmlformats.org/spreadsheetml/2006/main}'
        with ZipFile(xlsx) as z:
            strings=[''.join(x.itertext()) for _,x in etree.iterparse(z.open('xl/sharedStrings.xml'),events=('end',),tag=ns+'si')]
            rows=[]; total=0; counts=collections.Counter(); headers=None
            for _,row in etree.iterparse(z.open('xl/worksheets/sheet1.xml'),events=('end',),tag=ns+'row'):
                values=[]
                for c in row:
                    letters=''.join(v for v in c.get('r','') if v.isalpha()); idx=0
                    for v in letters: idx=idx*26+ord(v)-64
                    while len(values)<idx: values.append(None)
                    v=c.find(ns+'v')
                    if v is not None:
                        value=v.text
                        if c.get('t')=='s': value=strings[int(value)]
                        elif c.get('t')=='b': value=bool(int(value))
                        elif c.get('t') not in ['str','e']:
                            try: value=float(value); value=int(value) if value.is_integer() else value
                            except (TypeError,ValueError): pass
                        values[idx-1]=value
                    elif c.get('t')=='inlineStr': values[idx-1]=''.join(c.xpath('.//*[local-name()="t"]/text()'))
                if headers is None: headers=values; fuel_idx=headers.index('fuel_type')
                elif any(v is not None for v in values):
                    total+=1; fuel=values[fuel_idx] if len(values)>fuel_idx else None; counts[str(fuel)]+=1
                    if str(fuel).strip().casefold()=='electric':
                        values += [None]*(len(headers)-len(values)); rows.append(values[:len(headers)]+[int(row.get('r'))])
                row.clear()
                while row.getprevious() is not None: del row.getparent()[0]
        raw=pd.DataFrame(rows,columns=headers+['source_excel_row'])
        profile=dict(total_rows=total,electric_rows=len(raw),fuel_counts=dict(counts),columns=headers,source_filename=xlsx.name,source_sha256=hashlib.sha256(xlsx.read_bytes()).hexdigest())
        raw.to_csv(rawdir/'electric_source_snapshot.csv',index=False)
        (rawdir/'source_profile.json').write_text(json.dumps(profile,indent=2))
        return raw,profile
    if csv.exists():
        full=pd.read_csv(csv); full['source_excel_row']=range(2,len(full)+2)
        # source_excel_row here refers to the corresponding header-inclusive CSV row.
        raw=full[full.fuel_type.fillna('').str.strip().str.casefold().eq('electric')].copy()
        profile=dict(total_rows=len(full),electric_rows=len(raw),columns=[c for c in full if c!='source_excel_row'],source_filename=csv.name,source_sha256=hashlib.sha256(csv.read_bytes()).hexdigest())
        raw.to_csv(rawdir/'electric_source_snapshot.csv',index=False)
        (rawdir/'source_profile.json').write_text(json.dumps(profile,indent=2))
        return raw,profile
    profile=json.loads((rawdir/'source_profile.json').read_text())
    raw=pd.read_csv(rawdir/'electric_source_snapshot.csv')
    assert len(raw)==profile['electric_rows'], 'Snapshot count differs from audited source profile'
    print('Using included Electric subset snapshot; full workbook counts come from its audited source profile.')
    return raw,profile


# Embedded M1 implementation: m1_pipeline.py
"""M1 reproducible cleaning, diagnostics and preprocessing comparison.
No predictive performance is reported on the held-out 20% test set.
"""
from pathlib import Path
import json, re, time
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import IsolationForest

NUMERIC = ['age', 'mileage', 'seller_rating', 'driver_rating']
BINARY = ['accidents_or_damage', 'one_owner', 'personal_use_only']
PREMIUM = ['Tesla','Porsche','BMW','Mercedes-Benz','Audi','Jaguar','Polestar','Lucid','Rivian','Genesis']
SEED = 42

def directories(root):
    root = Path(root)
    for d in ['data/raw','data/clean','data/diagnostics','figures','reports','notebooks','scripts']:
        (root/d).mkdir(parents=True, exist_ok=True)
    return root

def canonical_model(make, model):
    """Explicit model families; original trim text remains in model.
    The review flag identifies generic labels inferred from fuel/engine evidence.
    """
    u = str(model).upper()
    review = False
    if make == 'Tesla':
        match = re.match(r'MODEL\s+([3SXY])\b',u)
        line = 'Model '+match.group(1) if match else ('Roadster' if u.startswith('ROADSTER') else None)
    elif make == 'Audi':
        line = 'Q4 e-tron' if u.startswith('Q4') else ('e-tron GT' if 'GT' in u else 'e-tron')
    elif make == 'BMW': line = str(model).split()[0]
    elif make == 'Cadillac': line = 'LYRIQ' if u.startswith('LYRIQ') else None
    elif make == 'Chevrolet':
        line = 'Bolt EUV' if u.startswith('BOLT EUV') else ('Bolt EV' if u.startswith('BOLT EV') else ('Spark EV' if u.startswith('SPARK EV') else None))
    elif make == 'Ford':
        if u.startswith('MUSTANG MACH-E'): line = 'Mustang Mach-E'
        elif u.startswith('FOCUS'): line,review = 'Focus Electric', 'ELECTRIC' not in u
        elif u.startswith('F-150'): line,review = 'F-150 Lightning', 'LIGHTNING' not in u
        elif 'TRANSIT' in u: line,review = 'E-Transit', not u.startswith('E-TRANSIT')
        else: line = None
    elif make == 'GMC': line = 'HUMMER EV'
    elif make == 'Hyundai':
        line = 'IONIQ 5' if u.startswith('IONIQ 5') else ('IONIQ EV' if u.startswith('IONIQ EV') else ('Kona EV' if u.startswith('KONA EV') else None))
    elif make == 'Jaguar': line = 'I-PACE'
    elif make == 'Kia':
        line = 'Soul EV' if u.startswith('SOUL EV') else ('Niro EV' if u.startswith('NIRO') else ('EV6' if u.startswith('EV6') else None))
        review = u.startswith('NIRO') and not u.startswith('NIRO EV')
    elif make == 'Mazda': line = 'MX-30'
    elif make == 'Mercedes-Benz':
        line = 'B-Class Electric' if u.startswith('B-CLASS') else next((s for s in ['EQB','EQE','EQS'] if s in u),None)
    elif make == 'Mitsubishi': line = 'i-MiEV'
    elif make == 'Nissan': line = 'Leaf' if u.startswith('LEAF') else ('ARIYA' if u.startswith('ARIYA') else None)
    elif make == 'Porsche': line = 'Taycan'
    elif make == 'Subaru': line = 'Solterra'
    elif make == 'Toyota': line = 'RAV4 EV' if u.startswith('RAV4 EV') else ('bZ4X' if u.startswith('BZ4X') else None)
    elif make == 'Volkswagen': line = 'ID.4' if u.startswith('ID.4') else ('e-Golf' if u.startswith('E-GOLF') else None)
    elif make == 'Volvo':
        line = 'XC40 Recharge' if u.startswith('XC40') else ('C40 Recharge' if u.startswith('C40') else None)
        review = u.startswith('XC40') and 'RECHARGE' not in u
    else: line = None
    return (make+' '+line if line else make+' '+str(model), bool(review or line is None))

def clean_listings(raw, profile, root):
    root = directories(root)
    quality = []
    def log(step,before,after,reason):
        quality.append(dict(step=step,rows_before=int(before),rows_removed=int(before-after),rows_after=int(after),reason=reason))
    log('01 Electric filter',profile['total_rows'],len(raw),'Keep fuel_type Electric; extraction from supplied cars.xlsx')
    mask = raw.model.str.contains(r'Volt|Range Extender|ELR|Mirai|NEXO',case=False,na=False,regex=True)
    rejected = raw.loc[mask].copy()
    rejected['exclusion_reason'] = 'Explicit project non-BEV model rule: PHEV/range extender/hydrogen'
    rejected.to_csv(root/'data/diagnostics/non_bev_exclusions.csv',index=False)
    ev = raw.loc[~mask].copy()
    log('02 Non-BEV exclusions',len(raw),len(ev),'Explicit project non-BEV model-name rule; retain removal evidence')
    orig_cols = [c for c in raw.columns if c != 'source_excel_row']
    dup = ev.duplicated(subset=orig_cols,keep='first')
    ev.loc[dup].to_csv(root/'data/diagnostics/duplicate_exclusions.csv',index=False)
    n=len(ev); ev=ev.loc[~dup].reset_index(drop=True)
    log('03 Exact duplicates',n,len(ev),'Exact original-field duplicates; provenance column excluded from duplicate key')
    # Assign IDs once after cohort definition, before flags or train/test splitting.
    ev.insert(0,'listing_id',np.arange(len(ev),dtype=int))
    for c in orig_cols:
        if ev[c].dtype == object:
            ev[c]=ev[c].map(lambda v: re.sub(r'\s+',' ',v).strip() if isinstance(v,str) else v)
    ev=ev.drop(columns=['mpg','fuel_type'])
    drive = {'awd':'All-wheel Drive','all-wheel drive':'All-wheel Drive','rwd':'Rear-wheel Drive','rear-wheel drive':'Rear-wheel Drive','fwd':'Front-wheel Drive','front-wheel drive':'Front-wheel Drive','4wd':'Four-wheel Drive','four-wheel drive':'Four-wheel Drive','unknown':np.nan}
    ev['drivetrain']=ev.drivetrain.map(lambda v: drive.get(str(v).casefold(),v) if pd.notna(v) else np.nan)
    for c in ['year','mileage','seller_rating','driver_rating','driver_reviews_num','price','price_drop']+BINARY:
        ev[c]=pd.to_numeric(ev[c],errors='coerce')
    for c in BINARY:
        bad=ev[c].notna() & ~ev[c].isin([0,1])
        if bad.any(): raise ValueError('Unexpected binary values in '+c)
    for c in ['seller_rating','driver_rating']:
        bad=ev[c].notna() & ~ev[c].between(1,5)
        if bad.any(): raise ValueError('Unexpected rating scale in '+c)
    if (ev.price.isna() | (ev.price<=0)).any(): raise ValueError('Invalid price target requires separate review, never impute target')
    if (ev.year.isna() | (ev.year>2023)).any(): raise ValueError('Invalid year requires separate review')
    if (ev.mileage.dropna()<0).any(): raise ValueError('Negative mileage requires review')
    pairs=[canonical_model(m,x) for m,x in zip(ev.manufacturer,ev.model)]
    ev['model_line']=[x[0] for x in pairs]
    ev['model_line_review_flag']=[int(x[1]) for x in pairs]
    ev['age']=2023-ev.year
    ev['miles_per_year']=ev.mileage/ev.age.clip(lower=0.5)
    ev['high_mileage_flag']=np.where(ev.miles_per_year.isna(),np.nan,(ev.miles_per_year>15000).astype(int))
    ev['premium_brand_flag']=ev.manufacturer.isin(PREMIUM).astype(int)
    ev['price_drop_raw']=ev.price_drop.copy()
    ev['price_drop_was_missing']=ev.price_drop.isna().astype(int)
    ev['price_drop']=ev.price_drop.fillna(0) # explicit project assumption, not verified fact
    ev['initial_price']=ev.price+ev.price_drop # reconstructed, not original new-car price
    ev['price_cut_flag']=(ev.price_drop>0).astype(int)
    ev['log_price']=np.log(ev.price)
    ev['very_low_mileage_flag']=np.where(ev.mileage.isna(),np.nan,(ev.mileage<100).astype(int))
    ev=add_condition_features(ev)
    # Keep source uncertainties visible. This does not establish non-BEV status.
    generic=ev.model_line_review_flag.eq(1)
    evidence=ev.engine.fillna('').astype(str).str.contains(r'electric|\bac\b',case=False,regex=True,na=False)
    rex_engine=(ev.manufacturer.eq('BMW') & ev.model_line.eq('BMW i3') & ev.engine.fillna('').astype(str).str.contains(r'0\.65\s*L|647\s*cc',case=False,regex=True))
    ev['bev_status_review_flag']=((generic & ~evidence) | rex_engine).astype(int)
    ev['bev_status_review_reason']=np.where(rex_engine,'BMW i3 engine lists 0.65L: possible range extender; confirm source',np.where(generic & ~evidence,'Generic model label and missing/zero engine evidence','not_flagged'))
    log('04 Category and feature preparation',len(ev),len(ev),'Standardise drivetrains and model families; preserve gaps; preserve all price outliers')
    pd.DataFrame(quality).to_csv(root/'data/diagnostics/data_quality_log.csv',index=False)
    mapping=ev.groupby(['manufacturer','model','model_line','model_line_review_flag'],dropna=False).agg(listing_count=('listing_id','size'),engine_examples=('engine',lambda s:' | '.join(sorted(s.dropna().astype(str).unique())[:3]))).reset_index()
    mapping.to_csv(root/'data/clean/model_name_mapping_for_M2.csv',index=False)
    ev.groupby(['manufacturer','model_line']).agg(listing_count=('listing_id','size'),first_model_year=('year','min'),last_model_year=('year','max'),review_count=('model_line_review_flag','sum')).reset_index().to_csv(root/'data/clean/model_lines_for_M2.csv',index=False)
    ev[ev.bev_status_review_flag.eq(1)].to_csv(root/'data/diagnostics/bev_status_review.csv',index=False)
    ev.attrs['quality']=quality
    print('Clean cohort:',len(ev),'brands:',ev.manufacturer.nunique(),'model families:',ev.model_line.nunique(),flush=True)
    return ev

def missingness(ev,root):
    root=Path(root)
    reasons={
      'mileage':('MAR/MNAR hypotheses','Very few gaps; missing odometer information may depend on listing process. Mechanism cannot be determined from these data.'),
      'seller_rating':('MAR/MNAR hypotheses','Rates differ across observed seller/brand groups; seller size is not measured, so small sellers cannot be asserted as the cause.'),
      'driver_rating':('MAR/MNAR hypotheses','Inspect driver_reviews_num and model-year; no reviews and new models may lack ratings.'),
      'accidents_or_damage':('MAR/MNAR hypotheses','History unavailable/not disclosed are possible; missing is not equivalent to no accident.'),
      'one_owner':('MAR/MNAR hypotheses','History reporting may depend on seller/listing; unknown is not equivalent to multiple owners.'),
      'personal_use_only':('MAR/MNAR hypotheses','History reporting may depend on seller/listing; unknown use history must remain missing.'),
      'price_drop_raw':('Structural absence or MAR/MNAR; unverified','Project assumes absent recorded reduction = zero. Original values and missing indicator retained; no causal or future-price-cut claim.'),
      'engine':('MAR/MNAR hypotheses','Specifications not supplied or not extracted; no evidence to distinguish mechanisms.'),
      'drivetrain':('MAR/MNAR hypotheses','Not supplied, unknown, or not extracted; evaluate by model/seller; preserve missing.'),
      'exterior_color':('MAR/MNAR hypotheses','Optional seller description or extraction gap; mechanism unconfirmed.'),
      'interior_color':('MAR/MNAR hypotheses','Optional seller description or extraction gap; mechanism unconfirmed.'),
    }
    rows=[]
    for c in ev.columns:
        mechanism,why=reasons.get(c,('No missing values' if not ev[c].isna().any() else 'Undetermined','No filling at handover; downstream preprocessing fits only on training data.'))
        rows.append(dict(field=c,missing_count=int(ev[c].isna().sum()),missing_pct=100*ev[c].isna().mean(),mechanism_hypothesis=mechanism,explanation=why))
    result=pd.DataFrame(rows).sort_values('missing_pct',ascending=False)
    result.to_csv(root/'data/diagnostics/missingness_profile.csv',index=False)
    for group in ['manufacturer','seller_name']:
        rates=ev.assign(seller_rating_missing=ev.seller_rating.isna().astype(int),driver_rating_missing=ev.driver_rating.isna().astype(int)).groupby(group,dropna=False).agg(listings=('listing_id','size'),seller_rating_missing_pct=('seller_rating_missing',lambda s:s.mean()*100),driver_rating_missing_pct=('driver_rating_missing',lambda s:s.mean()*100)).reset_index()
        rates.to_csv(root/f'data/diagnostics/missingness_by_{group}.csv',index=False)
    ev.groupby('driver_reviews_num').agg(listings=('listing_id','size'),driver_rating_missing=('driver_rating',lambda s:s.isna().sum())).reset_index().to_csv(root/'data/diagnostics/rating_missing_vs_reviews.csv',index=False)
    return result

def mark_outliers(ev,root):
    root=Path(root)
    a=ev.groupby(['model_line','year']).log_price.agg(['size',lambda s:s.quantile(.25),lambda s:s.quantile(.75)])
    a.columns=['group_n','q1','q3']
    b=ev.groupby('model_line').log_price.agg(['size',lambda s:s.quantile(.25),lambda s:s.quantile(.75)])
    b.columns=['model_n','q1_model','q3_model']
    bounds=ev[['model_line','year']].join(a,on=['model_line','year']).join(b,on='model_line')
    small=bounds.group_n<10
    q1=bounds.q1.where(~small,bounds.q1_model); q3=bounds.q3.where(~small,bounds.q3_model)
    iqr=q3-q1
    ev['iqr_reference_n']=bounds.group_n.where(~small,bounds.model_n).astype(int)
    ev['iqr_reference_scope']=np.where(small,'whole_model_line','model_line_and_year')
    ev['iqr_outlier']=((ev.log_price<q1-1.5*iqr)|(ev.log_price>q3+1.5*iqr)).astype(int)
    # Descriptive full-cohort diagnostics only. NEVER supply these target-derived flags to a price model.
    x=SimpleImputer(strategy='median').fit_transform(ev[['log_price','mileage','age']])
    iso=IsolationForest(contamination=.01,random_state=SEED,n_jobs=1)
    ev['iso_outlier']=(iso.fit_predict(x)==-1).astype(int)
    ev['iso_anomaly_score']=iso.decision_function(x)
    designated=(ev.manufacturer.eq('Toyota') & ev.model_line.eq('Toyota bZ4X') & ev.price.eq(333333))
    ev['confirmed_error_flag']=designated.astype(int)
    ev['error_review_status']=np.where(designated,'project_designated_error_not_independently_verified',np.where(ev.iqr_outlier.eq(1)|ev.iso_outlier.eq(1),'statistical_flag_retained','not_flagged'))
    ev.loc[designated,['listing_id','source_excel_row','manufacturer','model','year','mileage','price','error_review_status']].assign(reason='Project instruction identifies Toyota bZ4X USD333333 as error; raw source alone cannot independently confirm').to_csv(root/'data/clean/outlier_ids.csv',index=False)
    review=ev[(ev.iqr_outlier==1)|(ev.iso_outlier==1)|designated].copy()
    review['review_decision']='Keep: unusual value alone is not proof of error'
    review.loc[designated,'review_decision']='Exclude from training only per project rule; retain in clean file and held-out test'
    review.loc[(review.mileage<100)&~designated,'review_decision']='Keep: near-new/demo vehicle; use very_low_mileage_flag'
    review.loc[review.model_line.eq('Tesla Roadster')&~designated,'review_decision']='Keep: possible collector vehicle; separate model family'
    review.to_csv(root/'data/diagnostics/outlier_review.csv',index=False)
    cross=pd.crosstab(ev.iqr_outlier,ev.iso_outlier).reindex(index=[0,1],columns=[0,1],fill_value=0)
    cross.to_csv(root/'data/diagnostics/outlier_method_crosstab.csv')
    pd.DataFrame([dict(method='Within-group log-price IQR',flagged=int(ev.iqr_outlier.sum()),removed_from_clean=0,decision='Primary explainable review screen; whole-model fallback for groups under 10'),dict(method='Isolation Forest',flagged=int(ev.iso_outlier.sum()),removed_from_clean=0,decision='Complementary multi-variable review; .01 contamination controls rate, not error certainty')]).to_csv(root/'data/diagnostics/outlier_method_comparison.csv',index=False)
    print('IQR flags',int(ev.iqr_outlier.sum()),'Isolation flags',int(ev.iso_outlier.sum()),'project-designated errors',int(designated.sum()),flush=True)
    return ev,cross

def make_imputation_pipeline(method):
    if method not in ['median','knn']: raise ValueError(method)
    imp=SimpleImputer(strategy='median') if method=='median' else KNNImputer(n_neighbors=5)
    numeric=Pipeline([('scale',StandardScaler()),('impute',imp)])
    # Treat binary history as binary. KNN averages must not invent fractional accident histories.
    history=Pipeline([('impute',SimpleImputer(strategy='most_frequent'))])
    prep=ColumnTransformer([('numeric',numeric,NUMERIC),('history',history,BINARY)])
    return Pipeline([('prep',prep),('ridge',Ridge(alpha=1.0))])

def compare_imputers(ev,root):
    root=Path(root)
    ids=ev.listing_id.to_numpy()
    dev_ids,test_ids=train_test_split(ids,test_size=.2,random_state=SEED)
    split=ev[['listing_id']].copy(); split['split']=np.where(split.listing_id.isin(test_ids),'held_out_test','development')
    split.to_csv(root/'data/clean/train_test_split_ids.csv',index=False)
    dev=ev[ev.listing_id.isin(dev_ids)].sort_values('listing_id').copy()
    cv=KFold(n_splits=5,shuffle=True,random_state=SEED)
    results=[]
    for method in ['median','knn']:
        for fold,(tr,va) in enumerate(cv.split(dev),1):
            t=time.time()
            train=dev.iloc[tr]; valid=dev.iloc[va]
            train=train[train.confirmed_error_flag.eq(0)]
            pipe=make_imputation_pipeline(method)
            pipe.fit(train[NUMERIC+BINARY],train.log_price)
            pred=pipe.predict(valid[NUMERIC+BINARY])
            results.append(dict(method=method,fold=fold,train_rows=len(train),validation_rows=len(valid),mae_log_price=mean_absolute_error(valid.log_price,pred),mae_usd=mean_absolute_error(valid.price,np.exp(pred)),seconds=time.time()-t))
            print('CV',method,fold,'log-MAE',round(results[-1]['mae_log_price'],6),'seconds',round(results[-1]['seconds'],1),flush=True)
            pd.DataFrame(results).to_csv(root/'data/diagnostics/imputation_cv_folds.csv',index=False)
    detail=pd.DataFrame(results)
    summary=detail.groupby('method').agg(mean_mae_log_price=('mae_log_price','mean'),sd_mae_log_price=('mae_log_price','std'),mean_mae_usd=('mae_usd','mean'),total_seconds=('seconds','sum')).reset_index()
    winner=summary.sort_values('mean_mae_log_price').iloc[0]['method']
    summary['selected']=summary.method.eq(winner)
    summary.to_csv(root/'data/diagnostics/imputation_cv_summary.csv',index=False)
    decision=dict(selected_method=winner,criterion='Lower mean 5-fold validation MAE on log price; development 80% only',development_rows=len(dev),held_out_test_rows=len(test_ids),test_performance_evaluated=False,numeric_fields=NUMERIC,binary_fields=BINARY,binary_imputation='most frequent for both methods',scaler_fit='training fold only, before numeric imputer',seed=SEED,knn_neighbors=5,ridge_alpha=1.0)
    (root/'data/diagnostics/imputation_decision.json').write_text(json.dumps(decision,indent=2))
    return summary,decision

def export_clean(ev,root):
    root=Path(root)
    ev.to_csv(root/'data/clean/ev_listings_clean_v2.csv',index=False)
    ev.describe().T.to_csv(root/'data/diagnostics/summary_statistics.csv')
    return root/'data/clean/ev_listings_clean_v2.csv'


## 1. Load and select Electric rows / 加载及筛选

The full source is scanned if supplied in data/raw; otherwise the audited Electric snapshot is used. The workbook was originally scanned in full, not sampled.

In [3]:
raw,profile=load_source(ROOT)
print('Full source rows:',profile['total_rows'])
print('Electric rows:',len(raw))
print(raw.head(3).to_string(index=False))


Using included Electric subset snapshot; full workbook counts come from its audited source profile.
Full source rows: 762091
Electric rows: 16192
manufacturer             model  year  mileage   engine      transmission      drivetrain fuel_type  mpg            exterior_color      interior_color  accidents_or_damage  one_owner  personal_use_only              seller_name  seller_rating  driver_rating  driver_reviews_num  price_drop  price  source_excel_row
        Audi RS e-tron GT Base  2022   1502.0 Electric 2-Speed Automatic All-wheel Drive  Electric  NaN      Suzuka Gray Metallic Black / Express Red                  0.0        1.0                1.0 Elliott Bay Auto Brokers            4.7            NaN                   0      6000.0 132880              8499
        Audi RS e-tron GT Base  2022  11020.0 Electric 2-Speed Automatic All-wheel Drive  Electric  NaN Daytona Gray Pearl Effect           Arras Red                  0.0        1.0                1.0       Audi South Atlanta   

## 2–3. True-EV exclusions, duplicates, categories and derived fields

Assign listing_id once, after cohort definition. Preserve raw model names; use explicit model-family rules rather than only the first token. The eight historically flagged BEV-status rows are retained by team decision (communicated 19 October 2026). The review item is closed; the flags remain for traceability and do not independently certify BEV identity.

In [4]:
ev=clean_listings(raw,profile,ROOT)
print(pd.read_csv(ROOT/'data/diagnostics/data_quality_log.csv').to_string(index=False))
print(ev.model_line.value_counts().to_string())
print('BEV status review IDs:',ev.loc[ev.bev_status_review_flag.eq(1),'listing_id'].tolist())


Clean cohort: 15155 brands: 19 model families: 44
                               step  rows_before  rows_removed  rows_after                                                                                 reason
                 01 Electric filter       762091        745899       16192                            Keep fuel_type Electric; extraction from supplied cars.xlsx
              02 Non-BEV exclusions        16192           962       15230                      Explicit project non-BEV model-name rule; retain removal evidence
                03 Exact duplicates        15230            75       15155         Exact original-field duplicates; provenance column excluded from duplicate key
04 Category and feature preparation        15155             0       15155 Standardise drivetrains and model families; preserve gaps; preserve all price outliers
model_line
Tesla Model 3                     2170
Nissan Leaf                       1741
Tesla Model S                     1631
Ford Mustang

## 4a. Profile missing values / 查看缺失

MAR/MNAR are hypotheses, not identified mechanisms. Seller size is not observed. Preserve unknown history rather than coding it as a clean car.

In [5]:
profile_missing=missingness(ev,ROOT)
print(profile_missing[profile_missing.missing_count>0].to_string(index=False))


                   field  missing_count  missing_pct                       mechanism_hypothesis                                                                                                                            explanation
          price_drop_raw           5614    37.043880 Structural absence or MAR/MNAR; unverified Project assumes absent recorded reduction = zero. Original values and missing indicator retained; no causal or future-price-cut claim.
           seller_rating           3539    23.352029                        MAR/MNAR hypotheses       Rates differ across observed seller/brand groups; seller size is not measured, so small sellers cannot be asserted as the cause.
           driver_rating           1630    10.755526                        MAR/MNAR hypotheses                                                 Inspect driver_reviews_num and model-year; no reviews and new models may lack ratings.
          interior_color           1231     8.122732                        

## 5. Review extreme observations / 检查异常

IQR uses the whole model family for sparse year groups. Isolation Forest is a complementary descriptive screen. These flags use asking price and must NEVER be predictive inputs. The designated Toyota error stays in the master file; M5 excludes it only from fitting data.

In [6]:
ev,crosstab=mark_outliers(ev,ROOT)
print(crosstab.to_string())
print(ev.loc[ev.confirmed_error_flag.eq(1),['listing_id','model','price','error_review_status']].to_string(index=False))


IQR flags 248 Isolation flags 152 project-designated errors 1
iso_outlier      0    1
iqr_outlier            
0            14771  136
1              232   16
 listing_id        model  price                                 error_review_status
      13975 bZ4X Limited 333333 project_designated_error_not_independently_verified


## 4b. Compare median/mode with KNN / 比较填补方法

Fixed 80/20 IDs; five shuffled folds on development only. Scaling and imputers fit within each training fold. Binary history uses mode in BOTH methods. Project-designated errors are excluded only from fold fitting, not fold validation. Pilot uses log-price MAE as primary score, USD MAE as secondary. No held-out predictive score is computed.

In [7]:
summary,decision=compare_imputers(ev,ROOT)
print(summary.to_string(index=False))
print(json.dumps(decision,indent=2))


CV median 1 log-MAE 0.361607 seconds 0.0
CV median 2 log-MAE 0.364795 seconds 0.0
CV median 3 log-MAE 0.363634 seconds 0.0
CV median 4 log-MAE 0.362364 seconds 0.0
CV median 5 log-MAE 0.364301 seconds 0.0
CV knn 1 log-MAE 0.361407 seconds 4.0
CV knn 2 log-MAE 0.365029 seconds 2.9
CV knn 3 log-MAE 0.363538 seconds 2.9
CV knn 4 log-MAE 0.362293 seconds 2.8
CV knn 5 log-MAE 0.364592 seconds 2.6
method  mean_mae_log_price  sd_mae_log_price  mean_mae_usd  total_seconds  selected
   knn            0.363372          0.001524  16618.353742      15.311068     False
median            0.363340          0.001331  16620.281556       0.135481      True
{
  "selected_method": "median",
  "criterion": "Lower mean 5-fold validation MAE on log price; development 80% only",
  "development_rows": 12124,
  "held_out_test_rows": 3031,
  "test_performance_evaluated": false,
  "numeric_fields": [
    "age",
    "mileage",
    "seller_rating",
    "driver_rating"
  ],
  "binary_fields": [
    "accidents_or_dam

## 6. Export and handover / 输出交接

Keep gaps in the exported CSV. Numeric target and price-derived diagnostics are not features. Reuse the split IDs in M5; do not select a favourable split. The method paragraph, dictionary and report are prepared in the delivered package.

In [8]:
clean_path=export_clean(ev,ROOT)
print('Saved:',clean_path)
assert ev.listing_id.is_unique
assert len(ev)==15155
assert ev.seller_rating.isna().any()
assert ev.confirmed_error_flag.sum()==1
print('Rows:',len(ev),'columns:',len(ev.columns),'unique IDs:',ev.listing_id.nunique())
print('Missing values preserved; statistical outliers retained.')


Saved: <local_project_root>/data/clean/ev_listings_clean_v2.csv
Rows: 15155 columns: 48 unique IDs: 15155
Missing values preserved; statistical outliers retained.


### Additional condition features (teacher feedback, 15 October)

We added seven transparent condition features: a low-mileage-for-age flag, a near-new flag, a clean-history flag, an equal-weight condition score, age and mileage bands, and a cumulative-use battery-wear proxy. Low mileage means annualised mileage below 7,500 miles, using a minimum age denominator of 0.5 years. Near-new means approximate age no greater than one year and mileage below 5,000 miles. Clean history means an explicit record of no accident or damage; missing history remains unknown. The condition score sums the three flags (0–3) and remains missing if any component is unknown. These definitions and cutoffs are project assumptions, not verified mechanical standards.

Age bands are 0–1, 2–3, 4–5 and 6+ years. Mileage bands are below 10,000, 10,000–below 30,000, 30,000–below 60,000 and 60,000+ miles. Battery-wear proxy is mileage divided by 10,000, representing cumulative use exposure only. The data contain no measured battery state of health, charging history or battery replacement history. This proxy cannot establish battery degradation, and is a linear rescaling of an existing predictor. Its added predictive benefit may therefore be zero. The old mileage-below-100 flag is preserved as very_low_mileage_flag.

All new features use fixed row-level rules and do not use prices, target statistics or full-sample quantiles. M5 reports that the with/without comparison has been completed on the agreed protocol. The condition features improved Ridge by 0.11 percentage points, mainly through age and mileage bands. XGBoost was 6.19% with versus 6.18% without, so the final model excludes them. These numbers are attributed to the team message, not recomputed in this cleaning notebook. Imputation and encoding must be fitted inside each training fold. Selection should use CV only; evaluate the held-out test after selection. Report log-scale MAE and dollar-scale MAE/RMSE/R² with units clearly distinguished. M1 created the features; M5 performed predictive evaluation. The initial log-MAE imputation pilot below is separate from the final model comparison and its selection metric. Hand-built scores are candidates, not validated vehicle inspections.


In [9]:
print(ev[CONDITION_FEATURES].head().to_string(index=False))
print(ev[CONDITION_FEATURES].isna().sum().to_string())

 low_mileage_for_age_flag  near_new_flag  clean_history_flag  condition_score  battery_wear_proxy age_band mileage_band
                      1.0            1.0                 1.0              3.0              0.1502      0-1       0-<10k
                      0.0            0.0                 1.0              1.0              1.1020      0-1     10k-<30k
                      1.0            0.0                 1.0              2.0              0.6741      0-1       0-<10k
                      0.0            0.0                 1.0              1.0              0.7822      0-1       0-<10k
                      1.0            1.0                 1.0              3.0              0.2319      0-1       0-<10k
low_mileage_for_age_flag      6
near_new_flag                 4
clean_history_flag          811
condition_score             817
battery_wear_proxy            6
age_band                      0
mileage_band                  6
